In [ ]:
#### imports and loading of cancer PAR train qrels, train queries, corpus, LLM prompt

import srsly
import re
import pandas as pd
import os
import json
from collections import defaultdict
from openai import OpenAI, APIConnectionError

In [ ]:

# sep='\t' to remove the tab spaces
train_qrels = pd.read_csv("../Task4_Cancer_Patients/Updated_PAR_cancer/new_PAR_qrels/new_cancer_qrels_train.tsv", sep='\t')                         #table showing which patients (query) matches with which article (corpus)

patient_info = pd.DataFrame(list(srsly.read_jsonl("../Task4_Cancer_Patients/Updated_PAR_cancer/new_PAR_queries/new_cancer_train_queries.jsonl")))    #contains the cancer patient id's and their patient info

articles = pd.DataFrame(list(srsly.read_jsonl("../Task4_Cancer_Patients/Updated_PAR_cancer/new_cancer_corpus.jsonl")))                               #contains the article id's, titles, and abstracts 

print("train_qrels:", train_qrels.shape)
print("patient_info:", patient_info.shape)
print("articles:", articles.shape)

display(train_qrels.head())
display(patient_info.head())
display(articles.head())

with open("hippo_questions_prompt.txt", "r", encoding="utf-8") as f:
    prompt_template = f.read()


In [ ]:
# make all IDs strings, just to avoid annoying mismatches
train_qrels["query_id"] = train_qrels["query_id"].astype(str)
train_qrels["corpus_id"] = train_qrels["corpus_id"].astype(str)

patient_info["_id"] = patient_info["_id"].astype(str)
articles["_id"] = articles["_id"].astype(str)

print(train_qrels.columns.tolist())
print(patient_info.columns.tolist())
print(articles.columns.tolist())

In [ ]:
#### filter qrels so only top article row remains for each patient

first_article_links = train_qrels.drop_duplicates(subset="query_id", keep="first").copy()

print("Rows in original qrels:", len(train_qrels))
print("Rows after taking first article per patient:", len(first_article_links))

display(first_article_links.head())

In [ ]:
#### merge on the patient text

patient_article_pairs = first_article_links.merge(
    patient_info,
    left_on="query_id",
    right_on="_id",
    how="left"
)

print("After merging patient info:", patient_article_pairs.shape)
display(patient_article_pairs.head())


#### merge on article title and abstract

patient_article_pairs = patient_article_pairs.merge(
    articles,
    left_on="corpus_id",
    right_on="_id",
    how="left",
    suffixes=("_patient", "_article")
)

print("After merging articles:", patient_article_pairs.shape)
display(patient_article_pairs.head())

In [ ]:
#### inspect column names, rename 

print(patient_article_pairs.columns.tolist())

final_pairs = patient_article_pairs[
    [
        "query_id",
        "corpus_id",
        "text_patient",
        "title",
        "text_article"
    ]
].copy()

final_pairs = final_pairs.rename(columns={
    "query_id": "patient_id",
    "corpus_id": "article_id",
    "text_patient": "patient_text",
    "text_article": "article_text"
})

display(final_pairs.head())

In [ ]:
# #### check for mising merges

# print("Missing patient text:", final_pairs["patient_text"].isna().sum())
# print("Missing article text:", final_pairs["article_text"].isna().sum())
# print("Missing article tisle:", final_pairs["title"].isna().sum())

# missing_rows = final_pairs[
#     final_pairs["patient_text"].isna() |
#     final_pairs["article_text"].isna()
# ]

In [ ]:
# write final_pairs into separate file

records = final_pairs.to_dict(orient = "records")
srsly.write_jsonl("merged_PAR_train.jsonl", records)

**Execute starting here**

In [ ]:
# read file that so that previous code does not need to be rerun

merged_pairs = pd.DataFrame(list(srsly.read_jsonl("merged_PAR_train.jsonl")))  


In [10]:
display(merged_pairs.head())

,patient_id,article_id,patient_text,title,article_text
0,8674405-1,29510273,A 36-year-old G4P2 premenopausal woman with a ...,Ovarian suppression failure during GnRH agonis...,In premenopausal women treated for breast canc...
1,8674685-1,30281871,"We report a case of a 45-year-old woman, a non...",Langerhans cell histiocytosis in adults: Advan...,Langerhans cell histiocytosis (LCH) is a rare ...
2,8675572-1,21380941,A 67-year-old Caucasian female presented to ou...,Colorectal cancer screening in women with endo...,Colorectal cancer (CRC) is the most common gas...
3,8675577-1,25765179,Our patient is a 78-year-old male with a past ...,Somatostatin Receptors 2A and 5 Are Expressed ...,Merkel cell carcinoma (MCC) is a rare high-gra...
4,8675583-1,30694442,A 64-year-old Caucasian male smoker with a hor...,The Challenge of Managing Bladder Cancer and U...,Bladder cancer is the fourth most common cance...


In [15]:
only_text = merged_pairs.drop(merged_pairs.columns[[0, 1]], axis = 1)

In [16]:
only_text.head()

,patient_text,title,article_text
0,A 36-year-old G4P2 premenopausal woman with a ...,Ovarian suppression failure during GnRH agonis...,In premenopausal women treated for breast canc...
1,"We report a case of a 45-year-old woman, a non...",Langerhans cell histiocytosis in adults: Advan...,Langerhans cell histiocytosis (LCH) is a rare ...
2,A 67-year-old Caucasian female presented to ou...,Colorectal cancer screening in women with endo...,Colorectal cancer (CRC) is the most common gas...
3,Our patient is a 78-year-old male with a past ...,Somatostatin Receptors 2A and 5 Are Expressed ...,Merkel cell carcinoma (MCC) is a rare high-gra...
4,A 64-year-old Caucasian male smoker with a hor...,The Challenge of Managing Bladder Cancer and U...,Bladder cancer is the fourth most common cance...


In [ ]:
### prepare LLM
BASE_URL = "https://llm1-compute.cms.hu-berlin.de/v1/"
OPENAI_API_KEY = "secret_but_not_used"
PROMPT_FILE = "hippo_questions_prompt.txt"

with open(PROMPT_FILE, "r", encoding="utf-8") as f:
    PROMPT_TEMPLATE = f.read()

INPUT_FILE = "merged_PAR_train.jsonl"
RAW_OUTPUT = "output.jsonl"
OUTPUT_FILE = "questions.jsonl"

client = OpenAI(base_url=BASE_URL, api_key=OPENAI_API_KEY)

In [ ]:
def generate_questions(patient_text, article_title, article_text):
    prompt = PROMPT_TEMPLATE.format(
        patient_text=patient_text or "",
        article_title=article_title or "",
        article_text=article_text or ""
    )

    try:
        response = client.chat.completions.create(
            model="llm1",
            messages=[
                {"role": "system", "content": "You are a medical researcher. Output only the final questions, without reasoning or explanations."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=600,
            temperature=0.1,
            stream=False
        )
        return response.choices[0].message.content.strip()
    except APIConnectionError as e:
        return f"Error connecting to API: {e}"
    except Exception as e:
        return f"An error occurred: {e}"

In [ ]:
### raw llm output
with open(INPUT_FILE, 'r', encoding='utf-8') as f_in, \
    open(RAW_OUTPUT, 'w', encoding='utf-8') as f_raw:

    lines = f_in.readlines()[:3] 

    for line in lines:
        data = json.loads(line)
        
        patient_id = data.get('patient_id')
        patient_text = data.get('patient_text')
        article_id = data.get('article_id')
        article_title = data.get('title')
        article_text = data.get('article_text')
    
        raw_output = generate_questions(patient_text, article_title, article_text)

        result = {
            "patient_id": patient_id,
            "article_id": article_id,
            "questions": raw_output
        }
        
        f_raw.write(json.dumps(result, ensure_ascii=False) + '\n')
        f_raw.flush()

In [ ]:
### clean output to only extract questions
with open(RAW_OUTPUT, 'r', encoding='utf-8') as f_raw, \
     open(OUTPUT_FILE, 'w', encoding='utf-8') as f_clean:

    for line in f_raw:
        data = json.loads(line)
        raw_output = data.get('questions', '')

        # remove <think>
        cleaned = re.sub(r"<think>.*?</think>", "", raw_output, flags=re.DOTALL)

        # extract questions
        questions = re.findall(
            r"(?:\*\*Question:\*\*|Question\s*\d+:)\s*(.*?)(?=\n\s*\*|\n\s*\d+\.|\Z)",
            cleaned,
            flags=re.DOTALL
        )

        questions = [q.strip() for q in questions if q.strip()]

        result = {
            "patient_id": data.get('patient_id'),
            "article_id": data.get('article_id'),
            "questions": questions
        }

        f_clean.write(json.dumps(result, ensure_ascii=False) + '\n')
        f_clean.flush()

In [ ]:
### convert json to csv
data = []

with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        record = json.loads(line)
        # join questions list into single string
        record['questions'] = "; ".join(record.get('questions', []))
        data.append(record)

df = pd.DataFrame(data)

df.to_csv('questions.csv', index=False, sep=',', encoding='utf-8-sig')